### Enterprise Knowledge Assistant

#### Step 1- Document injestion

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("PROITBRIDGE_Employee_Handbook_2026_1.pdf")

documents = loader.load()

print("Total pages:", len(documents))

text = documents[0].page_content

print(text)
print(documents[0].metadata)

C:\Users\anand\AppData\Local\Temp\ipykernel_18188\3976613284.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total pages: 57
PEOPLE · POLICIES · PRACTICES
EmployeeHandbook
Everything you need to know about working at
PROITBRIDGE — how we hire, how we pay, how we grow,
how we protect our people and our clients, and what we
expect from each other, every single day.
DOCUMENT Employee Handbook — All Employees
VERSION Version 6.0  |  Effective 1 April 2026
APPLIES TO All full-time, part-time, contract and intern staff in India
OWNER Human Resources Department, PROITBRIDGE
REVIEW CYCLE Annually, or on any change in applicable law
CLASSIFICATION Internal — Confidential
PROITBRIDGE TECHNOLOGIES PRIVATE LIMITED PIB-HR-HB-2026-V6
{'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': 'PROITBRIDGE_Employee_Handbook_2026_1.pdf', 'total_pages': 57, 'page': 0, 'page_label': '1'}


#### step 2- Text cleaning

In [2]:
import re
clean_text = re.sub(r'\s+', ' ', text).strip()

print(clean_text)

PEOPLE · POLICIES · PRACTICES EmployeeHandbook Everything you need to know about working at PROITBRIDGE — how we hire, how we pay, how we grow, how we protect our people and our clients, and what we expect from each other, every single day. DOCUMENT Employee Handbook — All Employees VERSION Version 6.0 | Effective 1 April 2026 APPLIES TO All full-time, part-time, contract and intern staff in India OWNER Human Resources Department, PROITBRIDGE REVIEW CYCLE Annually, or on any change in applicable law CLASSIFICATION Internal — Confidential PROITBRIDGE TECHNOLOGIES PRIVATE LIMITED PIB-HR-HB-2026-V6


In [3]:
### step 3 - Chunkings 

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

print(chunks[0].page_content)

Total chunks: 258
PEOPLE · POLICIES · PRACTICES
EmployeeHandbook
Everything you need to know about working at
PROITBRIDGE — how we hire, how we pay, how we grow,
how we protect our people and our clients, and what we
expect from each other, every single day.
DOCUMENT Employee Handbook — All Employees
VERSION Version 6.0  |  Effective 1 April 2026
APPLIES TO All full-time, part-time, contract and intern staff in India
OWNER Human Resources Department, PROITBRIDGE


### Embeddings

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print(len(vectors))
print(len(vectors[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

258
384


#### Vecter DB 

In [5]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

#### Retrival

In [6]:
question = "How many annual leave days are allowed?"

results = vectorstore.similarity_search(
    question,
    k=2
)

for doc in results:
    print(doc.page_content)

Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.
Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.


#### Reranking

In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

question = "How many annual leave days are allowed?"

results = vectorstore.similarity_search(question, k=3)

pairs = [
    (question, doc.page_content)
    for doc in results
]

scores = reranker.predict(pairs)

ranked_results = sorted(
    zip(scores, results),
    key=lambda x: x[0],
    reverse=True
)

for score, doc in ranked_results:
    print(score, doc.page_content)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2.7233999 Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.
2.7233999 Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.
2.7233999 Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive

In [8]:
context = "\n\n".join(
    doc.page_content
    for score, doc in ranked_results
)

#### LLM

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1. LLM
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

# 2. Define Prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the given context.

Context:
{context}

Question:
{question}

Answer:
""")

# 3. Create chain
chain = prompt | llm

# 4. Generate response
response = chain.invoke({
    "context": context,
    "question": question
})

print(response.content)

The context does not specify the total number of annual leave days allowed. However, it states that Earned Leave (EL / Privilege Leave) accrues at 1.5 days for every completed month of service and is credited at the end of each month. This means an employee earns 18 days of Earned Leave per year (1.5 days × 12 months). Additionally, Casual Leave (CL) and Sick Leave (SL) are mentioned but their exact amounts are not specified.


In [11]:
print("Answer:", response.content)

print("------------------------------------------------")

for score, doc in ranked_results[:3]:
    print(doc.metadata["source"])
    print("Page:", doc.metadata["page"] + 1)
    print("****************************************************")

Answer: The context does not specify the total number of annual leave days allowed. However, it states that Earned Leave (EL / Privilege Leave) accrues at 1.5 days for every completed month of service and is credited at the end of each month. This means an employee earns 18 days of Earned Leave per year (1.5 days × 12 months). Additionally, Casual Leave (CL) and Sick Leave (SL) are mentioned but their exact amounts are not specified.
------------------------------------------------
PROITBRIDGE_Employee_Handbook_2026_1.pdf
Page: 18
****************************************************
PROITBRIDGE_Employee_Handbook_2026_1.pdf
Page: 18
****************************************************
PROITBRIDGE_Employee_Handbook_2026_1.pdf
Page: 18
****************************************************


In [12]:
#### evaluation

In [13]:
question = "How many annual leave days?"

results = vectorstore.similarity_search(
    question,
    k=3
)

contexts = [
    doc.page_content
    for doc in results
]

answer = "Employees get 20 days of annual leave."

ground_truth = "Employees are eligible for 20 days."

In [14]:
!pip install ragas

In [15]:
from datasets import Dataset

data = {
    "question": [question],
    "contexts": [contexts],
    "answer": [answer],
    "ground_truth": [ground_truth]
}


dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['question', 'contexts', 'answer', 'ground_truth'],
    num_rows: 1
})


In [16]:
!pip install ragas datasets


C:\Users\anand\anaconda3\Lib\site-packages\instructor\client_gemini.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [17]:
%pip install -U langchain-community

In [ ]:
%pip install --upgrade pip

In [19]:
%pip install "langchain-community==0.3.31"

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.5 MB 3.3 MB/s eta 0:00:01
   ---------------- ----------------------- 1.0/2.5 MB 1.8 MB/s eta 0:00:01
   ---------------- ----------------------- 1.0/2.5 MB 1.8 MB/s eta 0:00:01
   -------------------- ------------------- 1.3/2.5 MB 1.5 MB/s eta 0:00:01
   ------------------------ --------------- 1.6/2.5 MB 1.4 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.5 MB 1.4 MB/s eta 0:00:01
   ------------------------------------- -- 2.4/2.5 MB 1.4 MB/s eta 0:00:01
   ------------------------------------- -- 2.4/2.5 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 1.2 MB/s  0:00:02
  Attempting uninstall: langchain-community

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-experimental 0.4.2 requires langchain-community<1.0.0,>=0.4.2, but you have langchain-community 0.3.31 which is incompatible.


In [20]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

C:\Users\anand\AppData\Local\Temp\ipykernel_18188\3340946060.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\anand\AppData\Local\Temp\ipykernel_18188\3340946060.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [21]:
from langchain_community.chat_models.vertexai import ChatVertexAI

In [22]:
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy
    ],
    llm=llm,
    embeddings=embeddings
)

print(result)

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

{'faithfulness': 0.0000, 'answer_relevancy': 0.9112}


In [23]:
print("QUESTION:", question)

print("\nRETRIEVED CONTEXT:")
for ctx in contexts:
    print(ctx)

print("\nLLM ANSWER:")
print(response.content)

print("\nGROUND TRUTH:")
print(ground_truth)

QUESTION: How many annual leave days?

RETRIEVED CONTEXT:
Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.
Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.
Leave year
The leave year runs from 1 January to 31 December. Employ

In [24]:
ground_truth = """
Earned Leave accrues at 1.5 days
for every completed month of service.
This equals 18 days per year.
"""

In [25]:
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=llm,
    embeddings=embeddings
)

print(result)

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

{'faithfulness': 0.0000, 'answer_relevancy': 0.9112}


In [26]:
print("ANSWER:")
print(answer)

print("\nCONTEXTS:")
for c in contexts:
    print(c)

ANSWER:
Employees get 20 days of annual leave.

CONTEXTS:
Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.
Leave year
The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-
rated to the number of completed months remaining in the year.
7.2 Earned Leave (EL / Privilege Leave)
Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each
month.
May be availed only after confirmation. It accrues during probation but cannot be taken until you are
confirmed.
Leave year
The leave year runs from 1 January to 31 December. Employ

In [27]:
question = "How does Earned Leave accrue?"

response = (prompt | llm).invoke({
    "context": context,
    "question": question
})

answer = response.content

print(answer)

Earned Leave (EL / Privilege Leave) accrues at 1.5 days for every completed month of service and is credited to your balance at the end of each month. It accrues during probation but may only be availed after confirmation.


In [28]:
dataset = Dataset.from_dict({
    "question": [question],
    "contexts": [contexts],
    "answer": [answer],
    "ground_truth": [ground_truth]
})

In [29]:
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=llm,
    embeddings=embeddings
)

print(result)

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'answer_relevancy': 0.9565}


In [30]:
# ==========================================
# ENTERPRISE KNOWLEDGE ASSISTANT - RAG
# ==========================================

# 1. IMPORTS

import os
import re

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from sentence_transformers import CrossEncoder

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy


# 2. PDF LOADING

loader = PyPDFLoader("PROITBRIDGE_Employee_Handbook_2026_1.pdf")

documents = loader.load()

print("Total Pages:", len(documents))


# 3. TEXT CLEANING

for doc in documents:
    doc.page_content = re.sub(
        r'\s+',
        ' ',
        doc.page_content
    ).strip()

print("Text Cleaning Completed")


# 4. CHUNKING

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))


# 5. EMBEDDINGS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Embedding Dimension:", len(vectors[0]))


# 6. VECTOR DATABASE

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="employee_handbook"
)

print("Documents Stored in ChromaDB")


# 7. RETRIEVAL

question = "How does Earned Leave accrue?"

results = vectorstore.similarity_search(
    question,
    k=3
)

print("\nRetrieved Documents:")

for doc in results:
    print(doc.page_content)


# 8. RERANKING

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

pairs = [
    (question, doc.page_content)
    for doc in results
]

scores = reranker.predict(pairs)

ranked_results = sorted(
    zip(scores, results),
    key=lambda x: x[0],
    reverse=True
)

print("\nReranked Documents:")

for score, doc in ranked_results:
    print("Score:", score)
    print(doc.page_content)


# 9. CONTEXT CREATION

contexts = list(dict.fromkeys(
    doc.page_content
    for score, doc in ranked_results
))

context = "\n\n".join(contexts)


# 10. LLM

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing from your .env file")

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# 11. PROMPT

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the given context.

Do not invent information.

If the answer is not available in the context,
say that the handbook does not specify it.

Context:
{context}

Question:
{question}

Answer:
""")


# 12. GENERATE ANSWER

chain = prompt | llm

response = chain.invoke({
    "context": context,
    "question": question
})

answer = response.content

print("\nFINAL ANSWER:")
print(answer)


# 13. CITATIONS

print("\nSOURCES:")

for score, doc in ranked_results:

    print("Source:", doc.metadata["source"])
    print("Page:", doc.metadata["page"] + 1)


# 14. RAGAS EVALUATION

ground_truth = """
Earned Leave accrues at 1.5 days
for every completed month of service and is
credited to the employee's balance at the end
of each month.
"""

data = {
    "question": [question],
    "contexts": [contexts],
    "answer": [answer],
    "ground_truth": [ground_truth]
}

dataset = Dataset.from_dict(data)


# 15. EVALUATION METRICS

result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy
    ],
    llm=llm,
    embeddings=embeddings
)

print("\nRAG EVALUATION RESULTS:")
print(result)

C:\Users\anand\AppData\Local\Temp\ipykernel_18188\1724167373.py:23: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\anand\AppData\Local\Temp\ipykernel_18188\1724167373.py:23: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


Total Pages: 57
Text Cleaning Completed
Total Chunks: 256


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimension: 384
Documents Stored in ChromaDB

Retrieved Documents:
CHAPTER 07 Leave Policy Every leave type, how much you get, how it accrues, and how to apply. 7.1 Leave Entitlement Summary Leave Type Entitlement per Year Accrual Carry Forward Encashable Earned / Privilege Leave (EL)18 days 1.5 days per completed month Up to 45 days Yes, on separation Casual Leave (CL) 8 days Credited upfront on 1 January No — lapses No Sick Leave (SL) 10 days Credited upfront on 1 January Up to 20 days No Maternity Leave 26 weeks On qualifying event Not applicable No
approval Not applicable No Leave year The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro- rated to the number of completed months remaining in the year. 7.2 Earned Leave (EL / Privilege Leave) Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each month. May be availed only after confirmation. It accrues during probation but cannot be

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Reranked Documents:
Score: 6.0925245
approval Not applicable No Leave year The leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro- rated to the number of completed months remaining in the year. 7.2 Earned Leave (EL / Privilege Leave) Accrues at 1.5 days for every completed month of service; credited to your balance at the end of each month. May be availed only after confirmation. It accrues during probation but cannot be taken until you are confirmed. Minimum block of 0.5 day; there is
Score: 2.639904
CHAPTER 07 Leave Policy Every leave type, how much you get, how it accrues, and how to apply. 7.1 Leave Entitlement Summary Leave Type Entitlement per Year Accrual Carry Forward Encashable Earned / Privilege Leave (EL)18 days 1.5 days per completed month Up to 45 days Yes, on separation Casual Leave (CL) 8 days Credited upfront on 1 January No — lapses No Sick Leave (SL) 10 days Credited upfront on 1 January Up to 20 days No Maternity Leave 26

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]


RAG EVALUATION RESULTS:
{'faithfulness': 1.0000, 'answer_relevancy': 0.8329}


In [33]:
from ragas.metrics import context_precision
from ragas.metrics import context_recall

C:\Users\anand\AppData\Local\Temp\ipykernel_18188\1347344191.py:1: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision
C:\Users\anand\AppData\Local\Temp\ipykernel_18188\1347344191.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall


In [34]:
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=llm,
    embeddings=embeddings
)

print(result)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 1.0000, 'answer_relevancy': 0.8524, 'context_precision': 1.0000, 'context_recall': 1.0000}


In [35]:


!pip install streamlit